# Single Cell Scoring Metric Based on MV
*For Cai Lab **NanoMAP data analysis workflow**. Created by: Ellie Lai. This version was edited on 2026/08/05, and tested with **Python 3.13.5.***

**This script calculates the score for a single cell comparing the preference of MV tips of Receptor1 (R1) clusters over Receptor2 (R2) clusters.**

The template script processes results from **NearbySurf_analysis.ipynb** of given datasets, and loads **NearbySurf_analysis3.xlsx**. The template script uses **Receptor1** and **Receptor2** for receptors to be analyzed, which are replaced with the names of receptors to generate a **dataset-specific script** for running.

* How to use:
  * 1. Run the script: **Cell_Scoring_set-parameters.ipynb** to set parameters to modify this template script to match the specific datasets you want to analyze. 
  * 2. Set the folder_path containing the data in the modified script and run the modified script. 



**Formula:**
$$
cellscore =\frac{ \sum_{i=1}^{n_1} P_{i}^{R1^+R2^+} + \sum_{j=1}^{n_2} P_{j}^{R1^+R2^-} + \sum_{k=1}^{n_3} P_{k}^{R1^-R2^+}}{n_1 + n_2 + n_3 + n_4}
$$ 
where $n_1, n_2, n_3\ and\ n_4$ are the number of microvilli (MV) of $R1^+R2^+, R1^+R2^-, R1^-R2^+,\ and\ R1^-R2^-$, correspondingly.   
 



With $D_{R1}$ and $D_{R2}$ being the distances of the furthest clusters on this MV, and $D_{MV}$ the length of the MV, we have

* $P_{i}^{R1^+R2^+} = \frac{D_{R1}}{D_{MV}} - \frac{D_{R2}}{D_{MV}}\qquad$    for the $i_{th}\ R1^+R2^+$ MV
* $P_{j}^{R1^+R2^-} = \frac{D_{R1}}{D_{MV}} - 0\qquad\qquad  $    for the $j_{th}\ R1^+R2^-$ MV
* $P_{k}^{R1^-R2^+} = 0 - \frac{D_{R2}}{D_{MV}}\qquad\qquad  $    for the $j_{th}\ R1^-R2^+$ MV

The generized formular for all populations of MV is:
$$
CellScore =\frac{ \sum_{i=1}^{n} P_{i}}{total\ number\ of\ MV}
$$
where
* $P_{i} = \frac{D_{R1}}{D_{MV}} - \frac{D_{R2}}{D_{MV}}\quad$    for the $i_{th}\ $ MV

## 1. User Input

In [ ]:
# Path to the folder containing the data (reads all .xlsx files in the folder and subfolders)
root_folder = '/my_data_folder'  # <-- replace this with the path to your data folder

## 2. Import Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import ast
from pathlib import Path

## 3. Functions

### 3.1. Load Data 
Define a function to get all .xlsx file names in a given folder

In [ ]:
def find_excel_files(root_folder='.'):
    """
    Finds all .xlsx files recursively within a root folder.
    """
    # 1. Define the root path
    root_path = Path(root_folder)
    
    # 2. Use glob() with a recursive pattern
    # The pattern '**/*.xlsx' means:
    #   ** : Match this directory and all subdirectories, recursively.
    #   *.xlsx : Match any file ending with .xlsx.
    excel_files = list(root_path.glob('**/*.xlsx'))
    
    # 3. Extract the full absolute path as a string for each file
    # file.resolve() gets the absolute path
    file_full_paths = [str(file.resolve()) for file in excel_files]
    
    return file_full_paths

### 3.2. Define function cellscore_MV 
For an input excel file, MV and receptor cluster information is read in from sheets in the NearbySurf_analysis3.xlsx file: 

**Sheet Names:**
* "MV ID Cluster Info"
* "Receptor1/Receptor2 ID Cluster Info"
* "MV Cluster IDs"
* "MV and Receptor Max Dist Ratios"
* "Per_Microvillus"

A new df is created with extracted information from the listed sheets and an additional column is added to calculate the MV tip preference (Col 12). The dataframe is saved as a separate Excel file called **MV_for_scoring.xlsx**. 

**Columns for new df:**
* col_1: ID -> MV ID
* col_2: MV_Distance_to_Cell_Max -> $ D_{mv}$
* col_3: Receptor1_cluster_count -> number of Receptor1 clusters in MV
* col_4: Receptor2_cluster_count -> number of Receptor2 clusters in MV
* col_5: Receptor1_Surf2_IDs -> IDs of Receptor1 clusters in the MV
* col_6: Receptor2_Surf2_IDs -> IDs of Receptor2 clusters in the MV
* col_7: Receptor1_Distance -> Distances of Receptor1 clusters in the MV
* col_8: Receptor2_Distance -> Distances of Receptor2 clusters in the MV
* col_9: Max_Receptor1_Distance -> $ D_Receptor1 $
* col_10: Max_Receptor2_Distance -> $ D_Receptor2 $
* col_11: Category -> BOTH, Receptor1 ONLY, Receptor2 ONLY
* col_12: MVtip_Preference_Receptor1_over_Receptor2 -> **$P_i$**

In [ ]:
def cellscore_MV(inFile):
    # Load all MV and Receptor info
    df_mv_all = pd.read_excel(inFile, sheet_name= "MV ID Cluster Info")
    df_Receptor1_all = pd.read_excel(inFile, sheet_name= "Receptor1 ID Cluster Info")
    df_Receptor2_all = pd.read_excel(inFile, sheet_name= "Receptor2 ID Cluster Info")
    
    # Load MV occupancy info 
    df_mv_ocp = pd.read_excel(inFile, sheet_name="MV Cluster IDs")
    df_mv_category = pd.read_excel(inFile, sheet_name="Per_Microvillus")
    # Get total number of MV of the cell
    nMV_all = len(df_mv_category)
    # Make sure df col for receptor list of IDs have the datatype list
    df_mv_ocp['Receptor1_Surf2_IDs'] = df_mv_ocp['Receptor1_Surf2_IDs'].apply(ast.literal_eval)
    df_mv_ocp['Receptor2_Surf2_IDs'] = df_mv_ocp['Receptor2_Surf2_IDs'].apply(ast.literal_eval)
    
    # Load double positive MV and Receptor Max Dist Ratios Sheet
    df_mv_d = pd.read_excel(inFile, sheet_name = "MV and Receptor Max Dist Ratios")
    
    # Create a new dictionary for storing new rows
    key_list = ['ID', 
                'MV_Distance_to_Cell_Max',  
                'Receptor1_cluster_count', 
                'Receptor2_cluster_count',  
                'Receptor1_Surf2_IDs', 
                'Receptor2_Surf2_IDs', 
                'Receptor1_Distance', 
                'Receptor2_Distance',
                'Max_Receptor1_Distance', 
                'Max_Receptor2_Distance', 
                'MVtip_Preference_Receptor1_over_Receptor2',
                'Category'] 
    dict_row = dict.fromkeys(key_list)

    # Create a list to store dictionaries of rows
    rows_list = []

    # Load data into dictionary by row for each MV ID
    for row in range(len(df_mv_all)):
        
        # New df of this row
        df_row = df_mv_all.iloc[row]
    
        # Read values
        mv_id = df_row.ID
        mv_d_max = df_row['Distance to cell (max)']
        nReceptor1 = df_mv_category[df_mv_category['Surf1'] == mv_id]['Receptor1_cluster_count'].iloc[0]
        nReceptor2 = df_mv_category[df_mv_category['Surf1'] == mv_id]['Receptor2_cluster_count'].iloc[0]
    
        Receptor1_ids = df_mv_ocp[df_mv_ocp['Surf1'] == mv_id]['Receptor1_Surf2_IDs'].iloc[0]
        Receptor2_ids = df_mv_ocp[df_mv_ocp['Surf1'] == mv_id]['Receptor2_Surf2_IDs'].iloc[0]
    
        # Receptor distances
        if len(Receptor1_ids) > 0:
            Receptor1_d_list = df_Receptor1_all[df_Receptor1_all['ID'].isin(Receptor1_ids)]['Distance to cell (max)'].tolist()
            Receptor1_max = max(Receptor1_d_list)
        else:
            Receptor1_d_list = []
            Receptor1_max = 0
    
        if len(Receptor2_ids) > 0:
            Receptor2_d_list = df_Receptor2_all[df_Receptor2_all['ID'].isin(Receptor2_ids)]['Distance to cell (max)'].tolist()
            Receptor2_max = max(Receptor2_d_list)
        else:
            Receptor2_d_list = []
            Receptor2_max = 0
    
        # Create a new dictionary for the row
        dict_row = dict.fromkeys(key_list)
    
        # Write values
        dict_row['ID'] = mv_id
        dict_row['MV_Distance_to_Cell_Max'] = mv_d_max
        dict_row['Receptor1_cluster_count'] = nReceptor1
        dict_row['Receptor2_cluster_count'] = nReceptor2
        dict_row['Receptor1_Surf2_IDs'] = Receptor1_ids
        dict_row['Receptor2_Surf2_IDs'] = Receptor2_ids
        dict_row['Receptor1_Distance'] = Receptor1_d_list
        dict_row['Receptor2_Distance'] = Receptor2_d_list
        dict_row['Max_Receptor1_Distance'] = Receptor1_max
        dict_row['Max_Receptor2_Distance'] = Receptor2_max
        dict_row['Category'] = df_mv_category[
            df_mv_category['Surf1'] == mv_id
        ].Category.iloc[0]
    
        # Calculate the MV tip preference of Receptor1 over Receptor2 for each MV (DP and SP)
        dict_row['MVtip_Preference_Receptor1_over_Receptor2'] = (Receptor1_max - Receptor2_max) / mv_d_max
    
        # Append dictionary to list
        rows_list.append(dict_row)
    
    df_mv_for_score = pd.DataFrame(rows_list)
    
    # Save data to the data folder
    p = Path(inFile)
    subfolder_path = p.parent
    result_save_full = subfolder_path / 'MV_for_scoring.xlsx'
    df_mv_for_score.to_excel(str(result_save_full))
    
    # Calculate the preference of MV tip to Receptor1 over Receptor2 for the whole cell
    cell_score = df_mv_for_score['MVtip_Preference_Receptor1_over_Receptor2'].sum() / nMV_all

## 4. Loop through all files in the folder and get a score for each

In [ ]:
file_list = find_excel_files(root_folder)
result_list = []
r_keys = ['Folder','Condition','File_Name', 'Cell_Score']

for file in file_list:
    # Get folder and file names
    # 1. Create a Path object
    p = Path(file)
    
    # 2. Extract the file name (including extension)
    file_name = str(p.name)
    
    # 3. Extract the directory path (subfolder path)
    subfolder_path = str(p.parent) 
    # Note: p.parent returns a Path object. Use str(p.parent) to get a string.
    parent_dir_of_subfolder = str(p.parent.parent)
    
    # create a dictionary for each file to store the result
    r_dict = dict.fromkeys(r_keys)
    # Write value 
    r_dict['Folder'] = subfolder_path
    r_dict['Condition'] = subfolder_path.replace(parent_dir_of_subfolder+'/', "")
    r_dict['File_Name'] = file_name
    r_dict['Cell_Score'] = cellscore_MV(inFile=file)
    result_list.append(r_dict)

In [ ]:
# sanity check for which files have been processed for a cell score
print(file_list)

## 5. Result: save and preview

In [ ]:
# Save the result in a dataframe 
df_folder_result = pd.DataFrame(result_list)
# Export the dataframe to an Excel file
df_folder_result.to_excel('folder_results.xlsx')

# Group the dataframe by condition and print the description
grouped = df_folder_result.groupby('Condition')
print(grouped.describe())